<a href="https://colab.research.google.com/github/abhinavnautiyalDS/Finwise-GenAI-Assistant/blob/main/finwise-genai-capstone/task-03-04-Rag/task_3%264_Rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing Required packages

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
!pip install -q langchain langchain-community langchain-huggingface langchain-google-genai sentence-transformers faiss-cpu pypdf ipywidgets huggingface_hub google-auth


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 23.1 MB/s eta 0:00:00


Importing libraries

In [5]:
import os
from google.colab import userdata, files
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
import ipywidgets as widgets
from IPython.display import display, clear_output
from transformers import pipeline


Set up Hugging Face API key

In [6]:
try:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get('HUGGINGFACE_API_KEY')
except Exception as e:
    print(f"Error retrieving Hugging Face API key: {e}")
    print("Please ensure 'HUGGINGFACE_API_KEY' is set in Colab Secrets (optional for local use).")
    raise

Defining Embedded model to generate embeddings

In [21]:


try:
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    print("✅ Using Hugging Face Embeddings (sentence-transformers/all-MiniLM-L6-v2).")
except Exception as e:
    print(f"❌ Error initializing Hugging Face embeddings: {e}")
    print("Check if the model is downloaded or available online.")
    raise



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Using Hugging Face Embeddings (sentence-transformers/all-MiniLM-L6-v2).


Initialize LLM with fallback to local model


In [22]:
try:
    # Try Gemini with API key
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.2)
    print("Initialized Gemini 2.0 Flash with API key.")
except Exception as e:
    print(f"Gemini authentication failed: {e}")
    print("Falling back to local Hugging Face LLM...")

    # Fallback to local Hugging Face LLM
    try:
        llm = pipeline("text-generation", model="google/flan-t5-small")
        print("Initialized local Hugging Face LLM (flan-t5-small) as fallback.")
    except Exception as local_e:
        print(f"Error initializing local LLM: {local_e}")
        print("LLM initialization failed. Check dependencies and internet connection.")
        raise

Initialized Gemini 2.0 Flash with API key.


Pdf Loader

In [23]:
# Upload PDF file
print("Please upload your PDF file (e.g., financial prospectus or compliance report):")
try:
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file uploaded")
    pdf_path = list(uploaded.keys())[0]
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    if not documents:
        raise ValueError("No content extracted from PDF")
except Exception as e:
    print(f"Error loading PDF: {e}")
    print("Ensure the uploaded file is a valid, text-based PDF (not scanned images).")
    raise

Please upload your PDF file (e.g., financial prospectus or compliance report):


Saving AbhinavNautiyal_InternshalaResume (1).pdf to AbhinavNautiyal_InternshalaResume (1) (1).pdf


Chunking

In [24]:
try:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
    )
    splits = text_splitter.split_documents(documents)
    if not splits:
        raise ValueError("No document chunks created")
except Exception as e:
    print(f"Error splitting documents: {e}")
    raise


Vector Store

In [25]:
# Create FAISS vector store
try:
    vectorstore = FAISS.from_documents(splits, embeddings)
except Exception as e:
    print(f"Error creating FAISS vector store: {e}")
    print("Check if embeddings were generated correctly.")
    raise

# Create basic retriever
try:
    retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})
except Exception as e:
    print(f"Error setting up retriever: {e}")
    raise

# Enhance with MultiQueryRetriever for multi-step retrieval
try:
    multi_retriever = MultiQueryRetriever.from_llm(
        retriever=retriever,
        llm=llm,
        include_original=True
    )
except Exception as e:
    print(f"Error setting up MultiQueryRetriever: {e}")
    raise


Set up conversation memory


In [26]:
try:
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True,
        output_key="answer"
    )
except Exception as e:
    print(f"Error setting up memory: {e}")
    raise

# Create the conversational retrieval chain
try:
    qa_chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=multi_retriever,
        memory=memory,
        return_source_documents=True,
        verbose=False
    )
except Exception as e:
    print(f"Error creating ConversationalRetrievalChain: {e}")
    raise


/tmp/ipython-input-515782298.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


Cheking output

In [28]:
def on_submit(change):
    question = text_input.value.strip()
    if question:
        try:
            result = qa_chain.invoke({"question": question})
            response = result["answer"]
            sources = [doc.metadata for doc in result["source_documents"]]
            clear_output(wait=True)
            print(f"**Question:** {question}")
            print(f"**Answer:** {response}")
            print(f"**Sources:** {sources}")
            display(widgets.VBox([text_input, button]))
            text_input.value = ""
        except Exception as e:
            print(f"Error processing question: {e}")
            print("Check API quotas, document content, or try a different question.")

# Create widgets
text_input = widgets.Text(
    value='',
    placeholder='Ask a question (e.g., "What are the key risks in this financial product?")',
    description='Question:',
    layout={'width': '80%'}
)
button = widgets.Button(description="Submit", button_style='primary')

# Bind button click to on_submit function
button.on_click(on_submit)

# Display widgets
display(widgets.VBox([text_input, button]))

**Question:** his schooling
**Answer:** Abhinav Nautiyal completed Senior Secondary (XII) in Science from Government Boys Sender Secondary School in 2019, achieving a percentage of 89.00%. He also earned a Bachelor of Science (B.Sc) in Physics from Rajdhani College (University of Delhi) between 2019 and 2022, with a percentage of 80.00%.
**Sources:** [{'producer': 'dompdf 2.0.4 + CPDF', 'creator': 'PyPDF', 'creationdate': '2025-08-19T12:00:36+05:30', 'moddate': '2025-08-19T12:00:36+05:30', 'source': 'AbhinavNautiyal_InternshalaResume (1) (1).pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, {'producer': 'dompdf 2.0.4 + CPDF', 'creator': 'PyPDF', 'creationdate': '2025-08-19T12:00:36+05:30', 'moddate': '2025-08-19T12:00:36+05:30', 'source': 'AbhinavNautiyal_InternshalaResume (1) (1).pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, {'producer': 'dompdf 2.0.4 + CPDF', 'creator': 'PyPDF', 'creationdate': '2025-08-19T12:00:36+05:30', 'moddate': '2025-08-19T12:00:36+05:30', 'source